# 《PythAPCS123》單元 6-6：迴圈控制變數的常見陷阱

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根第三十八部曲（第六章第六部曲）  
**對應教材**：吳邦一老師《PythAPCS123：Python 程式設計從 APCS 實作 1 級到 3 級》第 17.1 節 for 迴圈注意事項（第 77 頁）  

---

### 🌟 本單元學習導航地圖
歡迎各位程式冒險者來到第六章【迴圈結構與控制流程】的第六座核心里程碑——「名偵探除錯特訓館」！

在學會了 `for`、`while`、`break`、`continue` 與雙重巢狀迴圈之後，各位已經具備了指揮電腦重複運算的所有基本武藝。  
但是，許多初學同學在自學或解題時，常常會遇到以下令人抓狂的「靈異現象」：
- 「*明明在 for 迴圈裡面寫了 i += 3 想跳步，為什麼電腦完全不理我，下一輪還是乖乖只加 1？*」
- 「*倒數 range(5, 1, -1) 明明想倒數到 1，為什麼印出來永遠在 2 就煞車，關鍵的 1 跑去哪裡了？*」
- 「*迴圈跑完之後，去讀取控制變數 i，為什麼有時候會拿到意想不到的數字，甚至直接報錯崩潰？*」
- 「*寫了 while 迴圈之後，按下執行電腦就卡住當機、畫面一直滾動停不下來（無窮迴圈）！*」
- 「*寫了雙重迴圈，結果外層明明要跑 5 次，卻只跑了 1 次就莫名其妙結束了！*」

請放心！這不是電腦中邪，也不是你的資質問題，而是碰上了程式語言中赫赫有名的**「迴圈控制變數五大經典地雷陷阱」**！  
對於身邊沒有老師和同學可以隨時發問的自學者來說，**能夠看懂這些陷阱的底層成因，比單純寫出正確代碼更加重要一千倍！**  
本單元將化身為最嚴謹的除錯偵探，為大家逐一拆解迴圈背後的記憶體運作機制：

1. **6-6-1 `for` 迴圈內竄改計數變數無效的「自動校正陷阱」**
2. **6-6-2 左閉右開與負步進倒數的「差一邊界陷阱（Off-by-one Error）」**
3. **6-6-3 迴圈結束後計數變數的「殘留值與生命週期陷阱」**
4. **6-6-4 `while` 迴圈步伐迷航與條件永遠為真的「無窮死結陷阱」**
5. **6-6-5 巢狀迴圈控制變數的「命名覆蓋（Shadowing）與歸零位置陷阱」**

> 💡 **學習小叮嚀（嚴格零提前依賴）**：本單元**嚴格禁止使用任何串列容器（`list`）、列表生成式、字串切片（`[::-1]`）或自訂函數（`def`）**！所有除錯案例皆使用純變數與數值運算呈現最真實的執行軌跡。請跟隨標準五步驟（**說明 ➔ 範例 ➔ 填空 ➔ 練習 ➔ 挑戰**），為自己的程式碼打造百毒不侵的鋼鐵防護罩！


### 🚂 6-6-1 `for` 迴圈內竄改計數變數無效的「自動校正陷阱」

**生活比喻：高鐵列車的固定行車時刻表**  
想像你搭乘一列固定停靠 5 個車站的高鐵列車（第 0 站、第 1 站、第 2 站、第 3 站、第 4 站）。  
當列車停靠在第 1 站時，司機突發奇想，拿出隨身手錶把指針手動撥快了 3 個小時（這相當於在迴圈肚子裡寫 `i += 3`）；  
請問：當列車重新開動開往下一站時，高鐵會直接瞬間移動飛越到第 4 站嗎？  
絕不可能！高鐵中央控制中心的排程早已印死在行車時刻表上，下一站必定是原定的「第 2 站」！  
司機自己撥快手錶，只是欺騙了他自己的眼睛；等到了下一站，車站的電子大鐘（Python 直譯器）會無情地把司機的手錶強行校正回正確的時間！

**底層運作機制：迭代器（Iterator）的強制賦值特性**  
吳邦一老師在講義第 77 頁特別警惕大家：**在 `for i in range(...)` 迴圈內部手動修改 `i` 的值，是完全無法改變迴圈的執行次數與既定軌跡的！**  
請看以下令人困惑的代碼：
```python
for i in range(5):
    print("進入迴圈：i =", i)
    i += 3  # 初學者試圖讓 i 跳步
    print("手動修改後：i =", i)
```
執行結果會是什麼？  
- 第 0 回合：進入時 $i=0$；修改後變成 $3$。
- 第 1 回合：進入時 $i$ 竟然**又變回 1**！
為什麼會這樣？因為 `for` 迴圈在每一回合即將開始的瞬間，直譯器內部做的事情本質上是：**「從 `range` 隊列中取出下一個預定值，強制覆蓋賦值給 `i`！」**  
你在前一回合尾端把 `i` 改成 100、改成 999，下一回合開始時，`range` 依然會把既定的下一個數字（1, 2, 3...）硬生生塞進變數 `i` 裡，把你先前的竄改覆蓋得一乾二淨！

**正確的跳步做法是什麼？**  
1. **如果步伐固定**：直接使用 `range` 的第三個參數 step！例如想每次跳 3 步，直接寫 `range(0, 15, 3)`！
2. **如果步伐不固定（需根據條件動態跳步）**：此時 `for` 迴圈不適用，請果斷改用 **`while` 迴圈**！在 `while` 迴圈中，變數完全由你自由加減，電腦絕對不會擅自覆蓋！

**APCS 考試實務建議：**  
在 APCS 觀念題中，非常喜歡出「在 for 迴圈中修改控制變數，問最終輸出」的陷阱題！請牢記口訣：**「for 的步伐天注定，內部竄改全歸零！」** 絕不要上當！

In [ ]:
# [2] Code 範例區：展示 for 迴圈內修改變數 i 被強制校正的現象

print("=== 實驗：在 for 迴圈肚子裡手動寫 i += 2 ===")

for i in range(4):
    print("--> [回合開始] range 給定的 i =", i)
    
    # 試圖手動跳步
    i += 2
    print("    [手動竄改後] 當前縮排內看見的 i =", i)
    
print("=== 實驗結束 ===")
print("💡 觀察結論：下一回合開始時，i 依然按照原定順序（0, 1, 2, 3）前進，竄改完全徒勞無功！")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 正確的跳步走訪方式：
# 某位同學原本寫了以下錯誤程式，企圖印出 1 到 10 之間的所有奇數（1, 3, 5, 7, 9）：
# for i in range(1, 10):
#     print(i)
#     i += 1  # 錯誤！這行完全無法達到每次加 2 的跳步效果！
#
# 請協助這位同學將程式修改為正確寫法：
# 直接在 range 參數中設定步進值 step = 2，徹底擺脫無效的內部竄改！
#
# 請在空格處填入 range 的三個參數（起始值、結束值、步進值）：
# ==========================================

print("正確的奇數跳步走訪：")

# 填入起始值 1、結束值 11（包含到 9）、步進值 2
for num in range(1, ___, ___):
    print(num, end=" ")

print()


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 指定步長數列加總器：
# 輸入兩個正整數 n 與 k（第一行輸入 n，第二行輸入步長 k，保證 k >= 1）。
# 規則：
# 請從 1 開始出發，每次增加步長 k，直到數值大於 n 為止（走訪數列為 1, 1+k, 1+2k... <= n）。
# 請計算並輸出：
# 1. 成功走訪到的所有數值之「總和」
# 2. 總共走訪了「幾個數字」（步數）
# （兩數以空格隔開）
# 注意：請直接使用 range 的步進參數完成，切勿在迴圈內部手動竄改變數！
#
# 【公開測試資料 1】
# 輸入：
# 10
# 2
# 預期輸出：
# 25 5
# （說明：走訪數值為 1, 3, 5, 7, 9，總和為 25，共 5 個數字）
#
# 【公開測試資料 2】
# 輸入：
# 8
# 3
# 預期輸出：
# 12 3
# （說明：走訪數值為 1, 4, 7，總和為 12，共 3 個數字）
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：



In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 動態變速跳步模擬（while 迴圈的正確實戰）：
# 如果跳步的步伐「不是固定常數」，而是每一輪都動態改變呢？
# 第一行輸入正整數 limit（上限值）。
# 請使用 while 迴圈模擬以下動態變速過程：
# 控制變數 pos 初始為 1，步長 step 初始為 1。
# 只要 pos <= limit：
# 1. 印出當前 pos
# 2. pos 往前跨越 step（pos += step）
# 3. 步長每次翻倍（step *= 2）
# 最終輸出總共跨越了幾步（走訪了幾個數字）。
# 例如 limit = 10：
# 第 1 步：pos = 1，跨越 step 1，下一位置 2，步長變 2
# 第 2 步：pos = 2，跨越 step 2，下一位置 4，步長變 4
# 第 3 步：pos = 4，跨越 step 4，下一位置 8，步長變 8
# 第 4 步：pos = 8，跨越 step 8，下一位置 16（已超過 10）
# 印出 1 2 4 8，總步數輸出：4。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：



### 🎯 6-6-2 左閉右開與負步進倒數的「差一邊界陷阱（Off-by-one Error）」

**生活比喻：百米衝刺與倒數跨年晚會**  
想像你在參加 100 公尺賽跑。裁判在跑道旁豎立告示牌：「抵達 100 公尺前請止步！」  
結果選手跑到第 99 公尺處，看到終點線近在咫尺，卻誤以為不能踩上去，直接原地剎車！差了這關鍵的 1 公尺，金牌當場飛了！  
再想像跨年倒數晚會，主持人拿著麥克風帶領全場觀眾倒數：「5、4、3、2……」然後主持人突然閉嘴轉身離場！  
全場觀眾滿臉錯愕：「咦？最重要的『1』和『0！新年快樂！』呢？」  
這就是程式界最令人啼笑皆非、卻也是最高頻發生的**「差一錯誤（Off-by-one Error）」**！

**底層運作機制與左閉右開的數學死穴**  
Python 的 `range(start, stop, step)` 遵循嚴格的數理區間：**左閉右開 $[start, stop)$**。  
- **起點 `start`**：必定包含（閉區間）。
- **終點 `stop`**：**絕對不包含（開區間）！在到達 `stop` 之前的那一個合法數值就會嘎然而止！**

讓我們來盤點兩大經典翻車現場：
1. **順數邊界翻車**：  
   計算 1 加到 $n$ 的總和，如果寫成 `range(1, n)`，走訪的最後一個數字其實是 $n - 1$！最關鍵的壓軸數字 $n$ 根本沒被加到！  
   **黃金解法**：終點必須寫成 `n + 1`！
2. **負步進倒數翻車（超級大魔王）**：  
   想要從 5 倒數到 1（包含 1），初學者直覺寫成：
   ```python
   for i in range(5, 1, -1):
       print(i)  # 輸出只有：5, 4, 3, 2！數字 1 神隱了！
   ```
   為什麼？因為 `stop = 1` 是開區間，倒數時碰到 1 之前（也就是 2）就停下來了！  
   **黃金解法**：若要包含 1，終點必須再往下一位寫成 **`0`**（`range(5, 0, -1)`）；若要倒數包含到 0，終點必須寫成 **`-1`**（`range(5, -1, -1)`）！

**APCS 考試實務建議：**  
在 APCS 評測中，有超過 30% 的錯誤提交是因為 Off-by-one 邊界遺漏！寫完迴圈後，請務必用手指頭在腦海中模擬：「**最後跑的那一輪，變數到底是多少？有沒有涵蓋到題目要求的極限？**」

In [ ]:
# [2] Code 範例區：展示倒數邊界的差一錯誤與修正

print("--- 錯誤示範：range(5, 1, -1) 遺漏了 1 ---")
for i in range(5, 1, -1):
    print(i, end=" ")
print("<-- 慘劇：數字 1 消失了！\n")

print("--- 正確示範：range(5, 0, -1) 包含到 1 ---")
for i in range(5, 0, -1):
    print(i, end=" ")
print("<-- 完美：1 順利登場！\n")

print("--- 正確示範：range(5, -1, -1) 包含到 0 ---")
for i in range(5, -1, -1):
    print(i, end=" ")
print("<-- 包含 0 的完美倒數！")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 電梯下樓廣播器：
# 電梯從 10 樓下降到地下 1 樓（樓層標示依序為 10, 9, 8... 1, 0, -1）。
# 要求迴圈必須完整走訪到 -1 樓。
# 依據左閉右開原則，若要包含 -1，stop 參數必須設定為 -2！
#
# 請在空格處填入：
# 1. range 的 stop 終點參數（使得倒數能順利包含 -1）。
# 2. 負步進值。
# ==========================================

print("電梯下樓停靠樓層：")

# 起點 10，終點設定為 -2（包含 -1），步長 -1
for floor in range(10, ___, ___):
    print(floor, end=" ")

print("\n抵達地下停車場！")


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 倒數連續整數累加器（雙向邊界嚴密守護）：
# 輸入兩個整數 a 與 b（第一行輸入 a，第二行輸入 b，保證 a >= b）。
# 請計算從 a 倒數到 b 之間「所有整數的總和」（必須包含 a 與 b 本身！）。
# 例如 a = 5, b = 1，總和應為 5 + 4 + 3 + 2 + 1 = 15。
#
# 【公開測試資料 1】
# 輸入：
# 5
# 1
# 預期輸出：
# 15
# （說明：5 + 4 + 3 + 2 + 1 = 15，少算了 1 會得到錯誤答案 14）
#
# 【公開測試資料 2】
# 輸入：
# 7
# 3
# 預期輸出：
# 25
# （說明：7 + 6 + 5 + 4 + 3 = 25）
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：



In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 雙向鏡像對稱數列產生器：
# 輸入一個正整數 n（1 <= n <= 9）。
# 請使用兩個連續的迴圈，印出一串鏡像對稱的數列：
# 先從 1 遞增到 n，再從 n - 1 遞減回 1，所有數字以空格隔開。
# 例如 n = 4 時，輸出：1 2 3 4 3 2 1。
# （注意：頂峰的 n 只會出現一次，遞減時從 n - 1 開始倒數到 1，請特別注意兩個迴圈的開閉邊界！）
# 若輸入 n = 1，則僅輸出 1。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：



### 👻 6-6-3 迴圈結束後計數變數的「殘留值與生命週期陷阱」

**生活比喻：黑板上的最後一筆粉筆字**  
想像下課鐘聲響起，老師走出教室。教室黑板的右下角，依然清清楚楚留著老師剛才寫下的最後一個數字「10」。  
這時候，值日生如果直接指著黑板說：「看！這是下一節課要考的題目！」全班肯定會抗議：這只是上一節課留下來的「殘留痕跡」，根本不是新的指令！  
在 Python 中，迴圈結束後，那個負責帶路的計數變數（例如 `i`）並沒有隨著迴圈結束而魂飛魄散，**它會像幽靈一樣繼續留在記憶體裡！**

**底層運作機制與兩大直覺誤區**  
許多從其他程式語言（如 C/C++、Java）轉過來、或是剛學 Python 的同學，對迴圈結束後的變數 `i` 常常有以下兩大誤解：
1. **誤解一：以為結束後 `i` 等於 `stop` 數值**  
   ```python
   for i in range(1, 5):
       pass
   print(i)
   ```
   請問最後印出什麼？初學者常猜 5。**錯！答案是 4！**  
   因為迴圈最後一次被執行的回合是 `i = 4`；當 `range` 發現下一個是 5 時，它根本就「沒有進入迴圈」，所以 `i` 永遠停留在最後一次順利跑完的值（4）！
2. **誤解二：空迴圈引發的變數未定義崩潰（NameError 幽靈）**  
   如果 `range` 的範圍一開始就是空的（例如 `range(5, 2)` 或是 `range(0)`）：
   ```python
   for x in range(0):
       pass
   print(x)  # 直接大爆炸！引發 NameError: name 'x' is not defined
   ```
   因為迴圈連第 0 輪都沒進去，變數 `x` 根本從未在記憶體中被誕生過！這時在迴圈外冒然呼叫它，程式會立刻崩潰！
3. **中途 break 時的殘留值**：  
   若在中途被 `break` 踢出迴圈，控制變數會精確保留**「觸發 break 當時的那個數值」**，這在線性搜尋中非常有用，但前提是你必須確定迴圈確實至少執行過一次！

**APCS 考試實務建議：**  
在需要「記錄找到的答案」時，**請務必在迴圈外事先宣告一個專用變數（例如 `ans = -1`）**！一旦在迴圈內找到目標，把值賦予 `ans` 再 `break`。絕對不要在迴圈結束後冒險直接拿迴圈變數 `i` 來當作搜尋結果，否則一旦搜尋範圍為空，你的程式就會直接因為 `NameError` 拿到 RE（Runtime Error）零分！

In [ ]:
# [2] Code 範例區：展示迴圈結束後變數的殘留值

print("--- 實驗 1：正常跑完 range(1, 5) ---")
for i in range(1, 5):
    pass  # 什麼都不做，讓迴圈跑完
print("迴圈結束後，i 的殘留值是：", i, "（注意：是 4 而不是 5！）\n")

print("--- 實驗 2：中途 break 的殘留值 ---")
for k in range(1, 100):
    if k == 7:
        break
print("中途 break 後，k 的殘留值是：", k, "（精確停在觸發 break 的 7！）\n")

print("--- 實驗 3：安全設計——事前初始化專用變數 ---")
target_idx = -1  # 步驟一：事前設定預設值（安全保險）
for idx in range(1, 6):
    if idx * idx == 16:
        target_idx = idx
        break
print("搜尋平方等於 16 的數字位置：", target_idx)


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 安全搜尋防禦機制：
# 尋找 1 到 20 之間第一個其立方大於 500 的整數。
# 為了防禦找不到時引發錯誤，我們在迴圈前準備了 result = -1。
# 一旦找到，將數值存入 result 並立即 break。
#
# 請在空格處填入：
# 1. 條件滿足時將當前變數 i 賦予 result 的陳述句。
# 2. 提前跳離迴圈的關鍵字。
# ==========================================

result = -1  # 安全防禦預設值

for i in range(1, 21):
    if i ** 3 > 500:
        # 1. 記錄答案
        result = ___
        # 2. 提前中斷
        ___

print("第一個立方大於 500 的整數是：", result)


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 尋找平方超越門檻的最小整數：
# 輸入一個正整數 m（10 <= m <= 1000）。
# 請在 1 到 100 之間依序尋找「第一個滿足 i * i > m」的最小整數 i。
# 一旦找到，請記錄該整數與其平方值，並立即中斷迴圈。
# 輸出說明：
# 請輸出該整數以及其平方值（兩數以空格隔開）。
# （請使用安全變數記錄答案，不要依賴迴圈變數在結束後的殘留狀態）
#
# 【公開測試資料 1】
# 輸入：
# 50
# 預期輸出：
# 8 64
# （說明：7*7=49 <= 50，8*8=64 > 50，故為 8 與 64）
#
# 【公開測試資料 2】
# 輸入：
# 200
# 預期輸出：
# 15 225
# （說明：14*14=196 <= 200，15*15=225 > 200，故為 15 與 225）
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：



In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 連續整數倍數探測與存在性判別：
# 輸入三個正整數 low, high, divisor（第一行 low，第二行 high，第三行 divisor，保證 low <= high）。
# 請在 low 到 high 的範圍內（含兩端），尋找「第一個能夠被 divisor 整除的整數」。
# 規則：
# 1. 必須在迴圈外設置安全防禦變數 found_num = -1。
# 2. 若找到，記錄該數值並立即中斷。
# 3. 迴圈結束後，若有找到則印出該數值；若完全沒找到（即 found_num 依然為 -1），則印出 NONE。
# 例如 low = 10, high = 20, divisor = 7：
# 10 到 20 之間的第一個 7 的倍數是 14，輸出：14。
# 若 low = 10, high = 13, divisor = 7：
# 範圍內沒有 7 的倍數，輸出：NONE。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：



### 🌀 6-6-4 `while` 迴圈步伐迷航與條件永遠為真的「無窮死結陷阱」

**生活比喻：滾輪上的倉鼠與被膠帶黏死的手煞車**  
想像籠子裡有一隻正在透明滾輪上飛奔的倉鼠。滾輪上沒有終點線，只要倉鼠不停腳，滾輪就會以時速 30 公里永遠旋轉下去！  
再想像一輛失控的卡車，司機想要煞車，卻驚恐地發現手煞車被強效膠帶死死黏在「全速前進」的檔位上，剎車失靈！  
這就是初學程式設計最驚心動魄的一幕——**「無窮迴圈（當機死結）」**！  
當你的程式突然完全沒有回應、終端機黑底白字瘋狂狂飆、筆記本右上角旋轉的齒輪轉個不停時，十之八九就是踩進了這個深坑！

**底層運作機制：剖析三大無窮死結元凶**  
與 `for` 迴圈有 `range` 自動照料不同，`while` 迴圈是完全「純手動檔」。以下是自學者最容易誤踩的三顆地雷：
1. **元凶一：忘記寫更新步進（最常見的粗心！）**  
   ```python
   count = 1
   while count <= 5:
       print(count)
       # 慘劇：忘了寫 count += 1！count 永遠是 1，條件永遠成立，無限狂印 1！
   ```
2. **元凶二：更新方向搞反（南轅北轍悲劇）**  
   ```python
   num = 1
   while num <= 10:
       print(num)
       num -= 1  # 慘劇：原本要往 10 前進，卻一路向負數狂奔！離 10 越來越遠！
   ```
3. **元凶三：浮點數精確比對的致命陷阱（0.1 + 0.2 != 0.3）**  
   ```python
   x = 0.0
   while x != 1.0:
       x += 0.1  # 慘劇：因為二進位浮點數精度誤差，x 永遠只會是 0.9999999999999999 或 1.0000000000000002，永遠踩不到 1.0，直接陷入無窮迴圈！
   ```

**自學救命秘笈：**  
1. **如果不幸當機卡死怎麼辦？**  
   在 Colab 或本機終端機中，請毫不猶豫立刻按下 **`Ctrl + C`**（或點擊 Colab 上方的「中斷執行階段」按鈕），強制把失控的直譯器拉回現實！
2. **浮點數終止條件防禦**：永遠使用 `<` 或 `>` 進行範圍比較（例如 `while x < 1.0:`），**絕對嚴禁在浮點數上使用 `!=` 或 `==` 作為迴圈條件**！

**APCS 考試實務建議：**  
在 APCS 評判中，無窮迴圈會導致程式在第一筆測資就因為超過 1 秒限制而領到「TLE（Time Limit Exceeded）」！提交程式碼前，務必對每個 `while` 迴圈進行「邊界與更新步伐」的嚴格健檢！

In [ ]:
# [2] Code 範例區：展示安全受控的 while 迴圈 vs 浮點數精度防禦

print("=== 正確的安全整數 while 迴圈 ===")
val = 1
while val <= 4:
    print("當前數值：", val)
    val += 1  # 確實向終點邁進
print("安全抵達終點！\n")

print("=== 浮點數安全防禦示範 ===")
f_val = 0.0
# 安全防禦：使用 < 而不是 != 進行判斷！
while f_val < 0.35:
    print("浮點數安全累加：", round(f_val, 2))
    f_val += 0.1
print("浮點數成功安全脫離，避免無窮死結！")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 拯救崩潰邊緣的失控除法迴圈：
# 某位同學原本寫了以下程式，想模擬數字被 2 整除削弱的過程，直到數字小於等於 1：
# n = 16
# while n > 1:
#     print(n)
#     # 慘劇：同學忘記寫 n //= 2，導致程式陷入死循環當機！
#
# 請協助補齊內部更新步伐，並確保使用整數除法：
# ==========================================

n = 16
step_count = 0

print("開始安全削弱數值：")
while n > 1:
    print("當前 n =", n)
    # 填入整數除以 2 的複合賦值更新指令，讓 n 真正往終點縮小！
    n ___ 2
    step_count += 1

print("成功安全結束！共經過", step_count, "次除法，最終 n =", n)


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 烤箱安全降溫計時器：
# 第一行輸入當前烤箱初始溫度 temp（正整數）。
# 第二行輸入目標安全溫度 target（正整數，保證 temp >= target）。
# 第三行輸入每分鐘自然降溫的度數 drop（正整數，保證 drop >= 1）。
# 規則：
# 請使用 while 迴圈模擬降溫過程。
# 只要目前溫度「嚴格大於」target，每過 1 分鐘，溫度就下降 drop 度。
# 一旦溫度降到「小於或等於」target，即達成安全目標，迴圈結束！
# 輸出說明：
# 請輸出達成安全目標所花費的「總分鐘數」以及「最終的實際溫度」（兩數以空格隔開）。
#
# 【公開測試資料 1】
# 輸入：
# 100
# 25
# 15
# 預期輸出：
# 5 25
# （說明：0分:100度 ➔ 1分:85 ➔ 2分:70 ➔ 3分:55 ➔ 4分:40 ➔ 5分:25度 <= 25，需 5 分鐘，最終溫度 25）
#
# 【公開測試資料 2】
# 輸入：
# 50
# 50
# 10
# 預期輸出：
# 0 50
# （說明：一開始溫度 50 度即已達到目標 <= 50，迴圈執行 0 次，耗時 0 分鐘，最終溫度 50）
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：



In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 蓄水池注入與漏水守衛模擬：
# 水池目前有 initial 公升水。
# 水池容量上限為 capacity 公升（保證 capacity >= initial）。
# 每天早上，水管會注入 in_flow 公升水；
# 每天晚上，水池底部裂縫會漏掉 out_flow 公升水。
# 特殊規則：
# 1. 若早上注入後水總量已達到或超過 capacity，此時水池已滿溢，當天白天即成功達標（不再扣除當晚漏水）！
# 2. 若當晚漏水後水總量小於等於 0，水池乾涸，模擬亦終止。
# 3. 若 in_flow <= out_flow 且 initial < capacity，代表注入比漏水還慢，永遠無法蓄滿！請直接輸出 IMPOSSIBLE！
# 輸入四行整數：initial, capacity, in_flow, out_flow。
# 請計算水池在第幾天早上首度達成滿溢（輸出天數）；若不可能達成則輸出 IMPOSSIBLE。
# 例如 initial = 0, capacity = 10, in_flow = 5, out_flow = 2：
# 第 1 天早上 0+5=5 < 10，晚上 5-2=3
# 第 2 天早上 3+5=8 < 10，晚上 8-2=6
# 第 3 天早上 6+5=11 >= 10，滿溢達標！輸出：3。
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：



### 👥 6-6-5 巢狀迴圈控制變數的「命名覆蓋（Shadowing）與歸零位置陷阱」

**生活比喻：客廳裡的兩隻小黑狗與洗碗盤**  
想像你家裡同時養了一隻大拉布拉多跟一隻小博美，你卻偷懶把牠們兩隻都取名叫「小黑」。  
當你站在客廳大喊一聲：「小黑過來！（變數覆蓋）」大狗小狗同時撲向你，撞成一團，你根本搞不清楚現在到底在跟誰說話！  
再想像你要洗三疊盤子（每疊有 4 個盤子）。你的洗碗精盆子應該放在哪裡？  
如果你每洗一個盤子就把洗碗精整盆倒掉換新水（累加器放錯層級在內層內部洗白），你洗一整天永遠只能洗好最後那一個盤子！

**底層運作機制與兩大殺手級錯誤**  
在雙重與巢狀迴圈中，初學者最常因為以下兩大陷阱痛失分數：
1. **殺手一：變數名稱覆蓋（Variable Shadowing 災難）**  
   ```python
   for i in range(3):       # 外層原本預期跑 i = 0, 1, 2
       for i in range(3):   # 內層竟然也取名叫 i！慘劇誕生！
           print(i, end=" ")
       print()
   ```
   內層執行時，會把外層的 `i` 強行霸占；當內層跑完時，`i` 變成了 2；回到外層下一次評估時，外層的進度已經被內層撕得粉碎！  
   **黃金口訣**：外層叫 `i`、內層叫 `j`、第三層叫 `k`，**各層變數涇渭分明，嚴禁撞名！**
2. **殺手二：累加器歸零位置放錯層級（最隱蔽的邏輯 Bug！）**  
   請仔細比較以下兩種「計算各班總分」的寫法：
   - **正確寫法**：每換一個新班級（外層），班級總分 `class_sum = 0` 歸零一次；內層累加該班每位同學分數；外層印出該班總分。
   - **錯誤寫法 A**：把 `class_sum = 0` 寫在最最外層外面，結果第 2 班的總分包含了第 1 班的分數，越加越大！
   - **錯誤寫法 B**：把 `class_sum = 0` 寫在內層迴圈肚子裡，結果每讀一位同學撲滿就被倒空一次，最後只印出最後一位同學的分數！

**APCS 考試實務建議：**  
巢狀結構中，**「變數的初始化位置」決定了它的生命週期與統計範疇！**  
- 想算「全校所有人的大總和」 ➔ 放在所有迴圈的最外層！
- 想算「每個班級各自的小總和」 ➔ 放在外層迴圈肚子裡、內層迴圈的正上方！
- 想算「單一同學的各科總和」 ➔ 放在處理該同學的最貼身層級！

In [ ]:
# [2] Code 範例區：展示累加器歸零在不同層級的巨大差異

print("=== 正確示範：每個分組獨立計算小計 ===")
for group in range(1, 3):  # 走訪 2 個分組
    group_sum = 0          # 關鍵！每組出發前獨立歸零！
    
    for member in range(1, 4):  # 每組 3 名成員，得分分別為 10, 20, 30
        score = member * 10
        group_sum += score
        
    print("第", group, "組結算總分：", group_sum, "（正確！）")

print("\n=== 錯誤對比：若歸零寫在內層肚子裡 ===")
for group in range(1, 2):
    for member in range(1, 4):
        wrong_sum = 0      # 致命錯誤！每人進來都被清空！
        wrong_sum += member * 10
    print("最終結算：", wrong_sum, "（慘劇！只剩下最後一人的分數 30！）")


In [ ]:
# ==========================================
# [3] Code 填空題
# 任務說明：
# 修正巢狀迴圈中的變數撞名危機：
# 某位同學原本寫了以下有嚴重視覺干擾與邏輯風險的程式：
# for i in range(1, 4):
#     for i in range(1, 3):  # 撞名！
#         print(i)
#
# 請協助將內層變數名稱更改為 j，並在最內層印出座標 (i, j)：
# ==========================================

print("乾淨無衝突的雙層走訪：")

for i in range(1, 4):        # 外層使用 i
    for ___ in range(1, 3):    # 1. 內層改用獨立變數 j
        print("(", i, ",", ___, ")", end=" ")  # 2. 印出座標 (i, j)
    print()


In [ ]:
# ==========================================
# [4] Code 練習題
# 任務說明：
# 分組競賽得分結算系統（歸零位置實戰）：
# 第一行輸入一個正整數 g（代表競賽總共有 g 個小組）。
# 接下來依序處理這 g 個小組的成績：
# 每個小組的第一行輸入一個正整數 m（代表該小組有 m 位組員）；
# 接下來有 m 行，每行輸入一位組員的得分（整數）。
# 請針對每一個小組，計算該組的總分並輸出在獨立的一行；
# 所有組別結算完畢後，最後一行輸出全場所有選手的「最高小組總分」。
#
# 【公開測試資料 1】
# 輸入：
# 2
# 3
# 10
# 20
# 30
# 2
# 40
# 50
# 預期輸出：
# 60
# 90
# 90
# （說明：第 1 組總分 10+20+30=60；第 2 組總分 40+50=90；全場最高小組總分為 90）
#
# 【公開測試資料 2】
# 輸入：
# 1
# 2
# 15
# 25
# 預期輸出：
# 40
# 40
# （說明：僅 1 組，該組總分 40，最高小組總分為 40）
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：



In [ ]:
# ==========================================
# [5] Code 挑戰題
# 任務說明：
# 矩陣行與列獨立加總驗證器：
# 輸入一個正整數 n（代表一個 n * n 的數值方陣，2 <= n <= 5）。
# 接下來有 n 行，每行依序輸入 n 個整數（以空格隔開）。
# 規則：
# 請計算每一橫列（Row）各自的總和，並輸出每一列的總和（每列一行）；
# 最後一行輸出所有橫列總和中的「最大值」。
# （注意：請確保每讀取新的一列時，列累加器 row_sum 必須精準歸零！不可將前一列的數值帶入下一列）
# 例如 n = 2：
# 輸入：
# 2
# 1 2
# 3 4
# 輸出：
# 3
# 7
# 7
#
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==========================================
# 請在下方撰寫你的程式碼：



---
### 🏆 恭喜完成單元 6-6，榮獲「程式除錯名偵探」金牌徽章！
### 🎊 恭喜掌握迴圈五大地雷的拆彈絕技，程式健壯性大幅躍升！

太令人敬佩了！在單元 6-6 中，各位自學者完成了最硬核、也最關鍵的一場「內功修練」。
身邊沒有老師隨時盯著，但你現在已經擁有了自行看破所有迴圈陷阱的法眼：
1. **for 迴圈不可竄改性**：明白迭代器的強行覆蓋機制，想要跳步一律使用 `range(..., step)` 或 `while`！
2. **左閉右開與倒數邊界**：掌握負步進倒數的 `-1` 終點偏移法，徹底告別差一錯誤（Off-by-one）！
3. **變數殘留值防禦機制**：學會在迴圈外預先建立安全防禦變數，避免 `NameError` 致命崩潰！
4. **while 迴圈死結拆彈**：熟記三大死結成因，學會用 `Ctrl+C` 自救與浮點數不等式防護！
5. **作用域與累加器歸零心法**：徹底杜絕變數撞名（Shadowing），精準把控「全域大總和」與「分組小統計」的歸零層級！

---

### 🗺️ 第六章學習進度導覽地圖（6 / 7 倒數衝刺！）：
- [x] **6.1 計數迴圈（for 搭配 range）**（range 單雙三參數、左閉右開與負步進倒數）
- [x] **6.2 迴圈累加器與計數器模式**（歸零初始化、逐回合數值累加、條件計數與擂台求極值）
- [x] **6.3 迴圈中斷與跳步（break, continue）**（提早破圈而出與略過本回合剩餘指令）
- [x] **6.4 條件迴圈（while 迴圈）**（未知次數迭代、輾轉相除法與位數逐位拆解）
- [x] **6.5 雙重與多重巢狀迴圈**（外層帶動內層、九九乘法表排版與幾何星號圖形）
- [x] **6.6 迴圈控制變數的常見陷阱**（走訪中竄改變數無效、倒數邊界失誤與無窮迴圈）
- [ ] **6.7 未知行數與檔案結尾測資讀取（while 搭配 EOF 處理）**（競技程式未知筆數讀取絕技）

---

👉 **下個單元預告：【單元 6-7 未知行數與檔案結尾測資讀取（while 搭配 EOF 處理）】**  
在 APCS 競賽或各大線上解題系統（ZeroJudge、Codeforces）中，常常會遇到一種令新手不知所措的題目描述：  
「*輸入有多行，每行兩個整數……直到檔案結束（EOF）為止！*」  
沒有告訴你總共有幾行！也沒有在結尾給你數字 0！面對這種「未知筆數的無限資料流」，我們該如何讓程式優雅讀取並在最後一刻安全收工？  
在第六章的最後一站，我們將解鎖競技程式設計的通關絕技——**EOF 讀取術**！第六章即將迎來大圓滿，敬請期待！
